In [2]:
import warnings
warnings.filterwarnings("ignore")

import os, pandas as pd, matplotlib.pyplot as plt, numpy as np, seaborn as sns, glob
from pynwb import NWBHDF5IO
from pathlib import Path
from convnwb.io import load_nwbfile

## example nwb directory and dataframes

In [4]:
sess_file = glob.glob('../../data/000623/sub-CS54/*.nwb')[0]
nwbfile, io = load_nwbfile(os.path.basename(sess_file), os.path.dirname(sess_file), return_io=True)
nwbfile

Data type,int64
Shape,"(603,)"
Array size,4.71 KiB
Chunk shape,None
Compression,None
Compression opts,None
Compression ratio,1.0
Data type,float64
Shape,"(603,)"
Array size,4.71 KiB
Chunk shape,None


## looping through neural data

In [36]:
df_neurs = pd.DataFrame(columns=['subj', 'sess', 'hemi', 'region', 'neur_id', 'base_start', 'base_stop', 'spike_times'])

sess_files = glob.glob('../../data/000623/sub-*/*.nwb')

for sess_file in sess_files:

    nwbfile, io = load_nwbfile(os.path.basename(sess_file), os.path.dirname(sess_file), return_io=True)

    df_trials = nwbfile.trials.to_dataframe()
    df_units = nwbfile.units.to_dataframe()[['unit_id_session', 'electrode_id', 'spike_times']]
    df_electrodes = nwbfile.electrodes.to_dataframe()

    # session info
    df_units['subj'], df_units['sess'] = nwbfile.subject.subject_id, nwbfile.session_id 
    df_units['base_start'], df_units['base_stop'] = df_trials['stop_time'].iloc[0], df_trials['start_time'].iloc[1]

    # neur info
    df_units = df_units.rename(columns={'unit_id_session': 'neur_id'})

    # electrode info
    df_units['electrode_id'] = df_units['electrode_id'].map(df_electrodes['location'])
    df_units['hemi'] = df_units['electrode_id'].str.split().str[0]
    df_units['region'] = df_units['electrode_id'].str.split().str[1]

    # sort
    df_units = df_units[['subj', 'sess', 'hemi', 'region', 'neur_id', 'base_start', 'base_stop', 'spike_times']]
    df_units = df_units.sort_values('region').reset_index(drop=True)

    df_neurs = pd.concat([df_neurs, df_units[['subj', 'sess', 'hemi', 'region', 'neur_id', 'base_start', 'base_stop', 'spike_times']]], ignore_index=True)

df_neurs.to_csv('../results/csvs/neuron_data.csv', index=False)
df_neurs

,subj,sess,hemi,region,neur_id,base_start,base_stop,spike_times
0,CS58,P58CSR1,Left,ACC,P58CS_R1_1_1_994_6,478.847432,531.161099,"[7.00682475, 10.84863625, 19.2006645, 19.27744..."
1,CS58,P58CSR1,Right,ACC,P58CS_R1_35_2_2425_5,478.847432,531.161099,"[8.0925435, 9.68744875, 11.07738625, 11.54073,..."
2,CS58,P58CSR1,Right,ACC,P58CS_R1_35_1_2195_5,478.847432,531.161099,"[8.01138725, 11.5975425, 13.11694775, 13.27707..."
3,CS58,P58CSR1,Right,ACC,P58CS_R1_34_4_3301_5,478.847432,531.161099,"[0.0645465, 0.19526525, 0.4629215, 0.7165465, ..."
4,CS58,P58CSR1,Right,ACC,P58CS_R1_34_3_3299_5,478.847432,531.161099,"[0.03082775, 0.10876525, 0.9559215, 2.45320175..."
...,...,...,...,...,...,...,...,...
1452,CS62,P62CSR1,Right,preSMA,P62CS_R1_18_3_3847_7,478.834214,531.051037,"[0.18107825, 0.7991095, 1.02176575, 1.20795325..."
1453,CS62,P62CSR1,Right,preSMA,P62CS_R1_18_2_3846_7,478.834214,531.051037,"[0.03289075, 0.1451095, 0.178672, 0.2311095, 0..."
1454,CS62,P62CSR1,Right,preSMA,P62CS_R1_18_1_3830_7,478.834214,531.051037,"[76.092017, 76.9399535, 79.0344525, 79.9236087..."
1455,CS62,P62CSR1,Right,vmPFC,P62CS_R1_7_1_791_9,478.834214,531.051037,"[2.0504835, 3.396921, 4.78307625, 7.4743565, 1..."


## looping through gaze data

In [3]:
from ephys_utills import get_et_timebins, compute_pixelperDVA, et_heatmap

SEC2MSEC = 1000.
nwb_input_dir = '../../data/000623/'

# ----- Basic metadata info about the video stimulus -----
frame_width, frame_height, vid_fps, nframes = 640, 480, 25.0, 11971
vid_fps, nframes = 29.97, 14351 # 11971/25*29.97

frame_duration_msec = SEC2MSEC / vid_fps
framesize = [frame_width, frame_height]

# define a duration to timebin the ET data
########################################################################################################################################################################## divided by vid_fps
timebin_sec = 1.0/vid_fps # define here as sec. used for saving results
timebin_msec = timebin_sec * SEC2MSEC # millisec.

nbins = np.round(frame_duration_msec * nframes / timebin_msec).astype(int)

nwb_session_files = sorted(glob.glob(os.path.join(nwb_input_dir, 'sub-*/*.nwb')))

# ----- Read ET (gaze) data from the NWB files -----
etdata_ses = []
pixel_dva_ses = []
missingdata_ratio = []
session_ids_et = []
select_subj_id, i = 0, 0


for session_ii in nwb_session_files:

    print(f'processing {os.path.basename(session_ii)}...')

    # Open the NWB file and read its content
    with NWBHDF5IO(session_ii,'r') as nwb_io: 
        nwbfile = nwb_io.read()
        
        session_ids_et.append(nwbfile.identifier)
        
        trials_df = nwbfile.trials.to_dataframe()
        enc_start_time = trials_df[trials_df['stim_phase']=='encoding']['start_time'].values[0]
        enc_stop_time = trials_df[trials_df['stim_phase']=='encoding']['stop_time'].values[0]
        
        gaze_data = nwbfile.processing['behavior']['EyeTracking']['SpatialSeries']
        gaze_xy = np.asarray(gaze_data.data)
        
        if gaze_data.rate is None:
            gaze_time = gaze_data.timestamps
        else:
            gaze_time = np.arange(0,len(gaze_xy))/(gaze_data.rate) + gaze_data.starting_time
        gaze_encoding = np.logical_and(gaze_time >= enc_start_time, 
                                        gaze_time <= enc_stop_time) 
    
        gaze_df = pd.DataFrame(data=np.c_[gaze_time[gaze_encoding],gaze_xy[gaze_encoding,:]],
                                columns=['RecTime','GazeX','GazeY'])    
    
        # get video display info and scale gaze to stimulus size
        display_info_raw = gaze_data.comments
        screen_wh, display_wh, display_area_i = display_info_raw.split('::')
    
        screen_w, screen_h = list(map(float,screen_wh.split('=')[1].split(',')))
        display_w, display_h = list(map(float,display_wh.split('=')[1].split(',')))
        display_area = list(map(float,display_area_i.split('=')[1].split(',')))
    
        _, pixel_dva_mean = compute_pixelperDVA([screen_w,screen_h])
        pixel_dva_ses.append(pixel_dva_mean)
    
        scale_dx = frame_width / display_w
        scale_dy = frame_height / display_h
    
        gaze_df['GazeX'] = (gaze_df['GazeX'] - display_area[0])*scale_dx 
        gaze_df['GazeY'] = (gaze_df['GazeY'] - display_area[1])*scale_dy 
    
        problem_inds_x = np.logical_not(gaze_df['GazeX'].astype(float).between(0,frame_width,
                                                                                inclusive='left'))
        
        problem_inds_y = np.logical_not(gaze_df['GazeY'].astype(float).between(0,frame_height,
                                                                                inclusive='left'))
        problem_inds = np.logical_or(problem_inds_x, problem_inds_y)
        gaze_df.loc[problem_inds,['GazeX','GazeY']] = np.nan
        
        # ###############################################################################################################################################
        # # do_op was originally None
        # # Downsample gaze data to the video frame rate
        # et_xy_binned = get_et_timebins(gaze_df, timebin_msec, do_op=None, 
        #                                 fix_length=True, nbins=nbins, keep_timebin_index=False,)
        
        # # sid's code
        # # et_xy_binned is a list of length nbins, where each entry is [array(GazeX), array(GazeY)]
        # # Convert to an object array of shape (nbins, 2)
        # et_arr = np.asarray(et_xy_binned, dtype=object)  # et_arr[t,0] is array of GazeX for bin t

        # # Now compute nanmean per bin for X and Y:
        # gaze_x_list = []
        # gaze_y_list = []

        # for t in range(nbins):
        #     arr_t = et_arr[t]             # get the full (n×2) array for bin t
        #     x_vals = arr_t[:, 0].astype(float)
        #     y_vals = arr_t[:, 1].astype(float)
        #     gaze_x_list.append(np.nanmean(x_vals))  # mean of valid X’s in bin t
        #     gaze_y_list.append(np.nanmean(y_vals))  # mean of valid Y’s in bin t

        # # gaze_x_list and gaze_y_list are now length=nbins, with NaNs only if the entire bin was NaN

        ## sid's code
        rec_time_list, gaze_x_list, gaze_y_list = gaze_df['RecTime'].tolist(), gaze_df['GazeX'].tolist(), gaze_df['GazeY'].tolist()

        df_sess = pd.DataFrame({
            'subj':  [nwbfile.subject.subject_id],
            'sess':  [nwbfile.session_id],
            'RecTime': [rec_time_list],
            'GazeX': [gaze_x_list],
            'GazeY': [gaze_y_list]
        })
        print(len(rec_time_list), len(gaze_x_list), len(gaze_y_list))
        df_sess.to_csv(f'../results/csvs/gaze/{nwbfile.session_id}_gaze.csv', index=False)

df_sess

processing sub-CS41_ses-P41CSR1_behavior+ecephys.nwb...
239428 239428 239428
processing sub-CS41_ses-P41CSR2_behavior+ecephys.nwb...
239429 239429 239429
processing sub-CS42_ses-P42CSR1_behavior+ecephys.nwb...
239427 239427 239427
processing sub-CS42_ses-P42CSR2_behavior+ecephys.nwb...
239423 239423 239423
processing sub-CS43_ses-P43CSR1_behavior+ecephys.nwb...
239425 239425 239425
processing sub-CS43_ses-P43CSR2_behavior+ecephys.nwb...
239429 239429 239429
processing sub-CS44_ses-P44CSR1_behavior+ecephys.nwb...
239420 239420 239420
processing sub-CS47_ses-P47CSR1_behavior+ecephys.nwb...
239017 239017 239017
processing sub-CS47_ses-P47CSR2_behavior+ecephys.nwb...
239018 239018 239018
processing sub-CS48_ses-P48CSR1_behavior+ecephys.nwb...
239424 239424 239424
processing sub-CS48_ses-P48CSR2_behavior+ecephys.nwb...
239410 239410 239410
processing sub-CS49_ses-P49CSR1_behavior+ecephys.nwb...
239425 239425 239425
processing sub-CS49_ses-P49CSR2_behavior+ecephys.nwb...
239423 239423 239423

,subj,sess,RecTime,GazeX,GazeY
0,CS62,P62CSR2,"[0.001, 0.003, 0.005, 0.007, 0.009000000000000...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
